In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_001 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_001_lgb_strict_raw_baseline.npy")
oof_002 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_002_cat_strict_raw_baseline.npy")
oof_003 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_003_lgb_compound_tyrelife_min.npy")
oof_004 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_004_lgb_lap_race_time_norm_min.npy")

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("001_lgb_raw", oof_001)
report("002_cat_raw", oof_002)
report("003_lgb_compound_tyrelife", oof_003)
report("004_lgb_lap_race_time_norm", oof_004)

pairs = [
    ("001", oof_001, "002", oof_002),
    ("001", oof_001, "003", oof_003),
    ("001", oof_001, "004", oof_004),
    ("002", oof_002, "003", oof_003),
    ("002", oof_002, "004", oof_004),
    ("003", oof_003, "004", oof_004),
]

for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

# simple average checks
blend_candidates = {
    "avg_001_004": (oof_001 + oof_004) / 2,
    "avg_002_004": (oof_002 + oof_004) / 2,
    "avg_003_004": (oof_003 + oof_004) / 2,
    "avg_001_002_004": (oof_001 + oof_002 + oof_004) / 3,
    "avg_001_002_003_004": (oof_001 + oof_002 + oof_003 + oof_004) / 4,
}

for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))

001_lgb_raw
  AUC: 0.940723081019204

002_cat_raw
  AUC: 0.9485020378244504

003_lgb_compound_tyrelife
  AUC: 0.9413109097188121

004_lgb_lap_race_time_norm
  AUC: 0.9416811403305019

001 vs 002: pearson=0.969599, spearman=0.963528
001 vs 003: pearson=0.993262, spearman=0.992558
001 vs 004: pearson=0.992932, spearman=0.990040
002 vs 003: pearson=0.971019, spearman=0.963512
002 vs 004: pearson=0.971108, spearman=0.964100
003 vs 004: pearson=0.990953, spearman=0.988319
avg_001_004 0.9416890502624848
avg_002_004 0.9470884168847017
avg_003_004 0.9421090580272116
avg_001_002_004 0.9457310574216122
avg_001_002_003_004 0.9450375015730456
